In [5]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

# Sample input DataFrame
data = pd.DataFrame({
    "text": [
        "The robot gazed at the stars, wondering about its creator.",
        "In the shimmering city of tomorrow, all was not well.",
        "Humanity's last hope was hidden in the ruins of the old world."
    ],
    "model": ["gpt-4", "llama-2", "gemini"],
    "temperature": [0.7, 1.0, 0.9],
    "batch_id": ["123", "123", "123"]
})

class LabellingBatch:
    text_area = widgets.Textarea(value='', description='Text:', layout=widgets.Layout(width="100%", height="500px"))
    usage_text = widgets.Combobox(value='', description='Usage:', layout=widgets.Layout(width="100%", height="20px"), ensure_option=False, options=['dialogue', 'exposition', 'opener', 'style'])
    model_label = widgets.Label(value='', layout=widgets.Layout(width="100%"))
    temperature_label = widgets.Label(value='', layout=widgets.Layout(width="100%"))
    batch_id_label = widgets.Label(value='', layout=widgets.Layout(width="100%"))
    rating = widgets.RadioButtons(
            options=["bad", "ok", "amazing"],
            value="bad",
            description='Rating:',
            layout=widgets.Layout(width="50%")
        )
    next_button = widgets.Button(description="Next", button_style='success')
    save_button = widgets.Button(description="Save labels", button_style='success')
    output = widgets.Output()

    input_df: pd.DataFrame
    outputs = []
    current_index: int
    
    def label_data(self, input_df: pd.DataFrame, experiment_name, current_index: int = 0):

        self.output_df: pd.DataFrame = pd.DataFrame()
        self.current_index = current_index
        self.input_df = input_df
        self.experiment_name = experiment_name
        
        # Attach event listener
        self.next_button.on_click(self.submit_and_next)
        self.save_button.on_click(self.write_labels)
        
        # Display initial data
        self.update_widgets(current_index)
        
        # Layout the widgets
        display(widgets.VBox([
            self.model_label,
            self.temperature_label,
            self.batch_id_label,
            self.text_area,
            self.usage_text,
            self.rating,
            self.next_button,
            self.save_button,
            self.output
        ]))

    def update_widgets(self, index: int):
        """Update widgets with the current row data."""
        df = self.input_df
        self.text_area.value = df.loc[index, "text"]
        self.usage_text.value = ""
        self.model_label.value = f"Model: {df.loc[index, 'model']}"
        self.temperature_label.value = f"Temperature: {df.loc[index, 'temperature']}"
        self.batch_id_label.value = f"Project: {df.loc[index, 'project_name']} Experiment: {df.loc[index, 'experiment_name']} Batch id: {df.loc[index, 'batch_id']}"
        self.rating.value = "bad"
    
    def submit_and_next(self, _):
        """Save current values and move to the next row."""
        new_row = {
            "target_text": self.text_area.value,
            "label": self.rating.value,
            "usage_text": self.usage_text.value,
            "input_index": self.current_index,
        }

        for column in ["text", "temperature", "model", "batch_id"]:
            new_row[column] = self.input_df.loc[self.current_index, column]

        self.outputs.append(new_row)
        
        # Increment index
        self.current_index += 1
        
        # Check if we reached the end
        if self.current_index < len(self.input_df):
            self.update_widgets(self.current_index)
        else:
            with output:
                clear_output()
                print("All entries have been labeled!")
        
        # Clear previous messages
        with output:
            clear_output()
            print(f"Entry {self.current_index}/{len(self.input_df)} labeled.")

    def write_labels(self, _):
        pd.DataFrame(self.outputs).to_parquet(f"labels/generate_writing/{self.experiment_name}.parquet")




In [13]:
import os

PROJECT_NAME = "QoGD"
EXPERIMENT_NAME = "initial"

folder = f"generated_text/{PROJECT_NAME}/{EXPERIMENT_NAME}/"
sorted(os.listdir(folder))


['.ipynb_checkpoints',
 '2024-12-28 13:58:21.757334',
 '2024-12-28 13:59:25.638082',
 '2024-12-29 15:47:43.865117',
 '2024-12-29 15:48:56.805075']

In [14]:
index = -2
with open(f"{folder}/{sorted(os.listdir(folder))[index]}/user_prompt.txt", 'r') as f:
    print(f.read()[:200])

### Beliefs explored

1. We live outside of Plato's cave, we live outside the simulation. There is no simulation - we already left it. We can go back in - it's warm and cozy inside. Womb, cocoon, and 


In [15]:
input_df = pd.read_parquet(f"{folder}{sorted(os.listdir(folder))[index]}/dataframe.parquet")
labeller = LabellingBatch()
labeller.label_data(input_df, PROJECT_NAME + "--" + EXPERIMENT_NAME)

In [9]:
len(labeller.outputs)


10

In [10]:
df = pd.read_parquet(f"labels/generate_writing/first_attempt.parquet")
df.size

80